In [1]:
import random
from rdkit import Chem
from molpher.core import MolpherMol, MolpherAtom
from molpher.core.morphing.operators import MorphingOperator
from rdkit.Chem.EnumerateStereoisomers import EnumerateStereoisomers, StereoEnumerationOptions
from rdkit.Chem import rdChemReactions
from rdkit.Chem import rdmolops
from rdkit.Chem import Descriptors  
from molpher.core import ExplorationTree as ETree

class AromaticHydroxylation(MorphingOperator):
    def __init__(self):
        super(AromaticHydroxylation, self).__init__()
        self._name = "Aromatic Hydroxylation (Phase I - Regioselective)"
        self._matches = []
        self.AROMATIC_C = Chem.MolFromSmarts("[c;H1]")

    def _get_para_score(self, mol, c_idx):
        """ Υπολογισμός para-θέσης σε 6μελείς δακτυλίους """
        for ring in mol.GetRingInfo().AtomRings():
            if c_idx in ring and len(ring) == 6:
                for r_idx in ring:
                    r_atom = mol.GetAtomWithIdx(r_idx)
                    has_ex_neighbor = any(n.GetIdx() not in ring for n in r_atom.GetNeighbors())
                    
                    if has_ex_neighbor:
                        path = Chem.GetShortestPath(mol, r_idx, c_idx)
                        if len(path) == 4: # Απόσταση para
                            return 10
        return 1 

    def setOriginal(self, mol):
        super(AromaticHydroxylation, self).setOriginal(mol)
        self._matches = []
        
        if not self.original: return
        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None: return

        if self.AROMATIC_C is not None:
            matches = rdkit_mol.GetSubstructMatches(self.AROMATIC_C)
            scored_sites = []
            for match in matches:
                c_idx = match[0]
                score = self._get_para_score(rdkit_mol, c_idx)
                scored_sites.append((c_idx, score))
                
            if scored_sites:
                max_score = max(site[1] for site in scored_sites)
                self._matches = [site[0] for site in scored_sites if site[1] == max_score]

    def morph(self):
        if not self.original: return None
        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None: return None

        if not self._matches:
            return MolpherMol(other=rdkit_mol)

        c_idx = random.choice(self._matches)
        
        try:
            rw_mol = Chem.RWMol(rdkit_mol)
            
            new_o_idx = rw_mol.AddAtom(Chem.Atom(8))
            c_atom = rw_mol.GetAtomWithIdx(c_idx)
            o_atom = rw_mol.GetAtomWithIdx(new_o_idx)
            
            c_atom.SetNoImplicit(False)
            c_atom.SetNumExplicitHs(0)
            o_atom.SetNoImplicit(False)
            o_atom.SetNumExplicitHs(0)
            
            rw_mol.AddBond(c_idx, new_o_idx, Chem.BondType.SINGLE)
            
            new_mol = rw_mol.GetMol()
            
            new_mol.UpdatePropertyCache(strict=False)
            Chem.SanitizeMol(new_mol, Chem.SanitizeFlags.SANITIZE_ALL)
            Chem.AssignStereochemistry(new_mol, cleanIt=True, force=True)
            
            return MolpherMol(other=new_mol)
        except:
            return MolpherMol(other=rdkit_mol)

    def getName(self): return self._name

hydroxylation_op = AromaticHydroxylation()

test_aromatic_molecules = {
    "1. Τολουένιο (Βενζολικός δακτύλιος -> Επιλογή Para θέσης)": "Cc1ccccc1",
    "2. Πυριδίνη (Ετεροαρωματικός δακτύλιος -> Επιτυχής Υδροξυλίωση χωρίς crash)": "c1ccncc1",
    "3. Κυκλοεξάνιο (Αλειφατικό -> Πρέπει να αγνοηθεί)": "C1CCCCC1"
}

print("=== STARTING AROMATIC HYDROXYLATION TESTING ===")
for name, smiles in test_aromatic_molecules.items():
    mol = MolpherMol(smiles)
    hydroxylation_op.setOriginal(mol)
    product = hydroxylation_op.morph()
    
    print(f"\n{name}")
    print(f"  SOURCE: {mol.getSMILES()}")
    print(f"  TARGET: {product.getSMILES() if product and product.getSMILES() != mol.getSMILES() else 'No change (Safe)'}")
print("\n===============================================")

=== STARTING AROMATIC HYDROXYLATION TESTING ===

1. Τολουένιο (Βενζολικός δακτύλιος -> Επιλογή Para θέσης)
  SOURCE: CC1=CC=CC=C1
  TARGET: CC1=CC=C(O)C=C1

2. Πυριδίνη (Ετεροαρωματικός δακτύλιος -> Επιτυχής Υδροξυλίωση χωρίς crash)
  SOURCE: C1=CC=NC=C1
  TARGET: OC1=CN=CC=C1

3. Κυκλοεξάνιο (Αλειφατικό -> Πρέπει να αγνοηθεί)
  SOURCE: C1CCCCC1
  TARGET: No change (Safe)



In [2]:
import random
from rdkit import Chem
from molpher.core import MolpherMol, MolpherAtom
from molpher.core.morphing.operators import MorphingOperator
from rdkit.Chem.EnumerateStereoisomers import EnumerateStereoisomers, StereoEnumerationOptions
from rdkit.Chem import rdChemReactions
from rdkit.Chem import rdmolops
from rdkit.Chem import Descriptors  
from molpher.core import ExplorationTree as ETree

class AromaticHydroxylation(MorphingOperator):
    def __init__(self):
        super(AromaticHydroxylation, self).__init__()
        self._name = "Aromatic Hydroxylation (Phase I - Regioselective)"
        self._matches = []
        self.AROMATIC_C = Chem.MolFromSmarts("[c;H1]")

    def _get_para_score(self, mol, c_idx):
        """ Υπολογισμός para-θέσης σε 6μελείς δακτυλίους """
        for ring in mol.GetRingInfo().AtomRings():
            if c_idx in ring and len(ring) == 6:
                for r_idx in ring:
                    r_atom = mol.GetAtomWithIdx(r_idx)
                    has_ex_neighbor = any(n.GetIdx() not in ring for n in r_atom.GetNeighbors())
                    
                    if has_ex_neighbor:
                        path = Chem.GetShortestPath(mol, r_idx, c_idx)
                        if len(path) == 4: # Απόσταση para
                            return 10
        return 1 

    def setOriginal(self, mol):
        super(AromaticHydroxylation, self).setOriginal(mol)
        self._matches = []
        
        if not self.original: return
        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None: return

        if self.AROMATIC_C is not None:
            matches = rdkit_mol.GetSubstructMatches(self.AROMATIC_C)
            scored_sites = []
            for match in matches:
                c_idx = match[0]
                score = self._get_para_score(rdkit_mol, c_idx)
                scored_sites.append((c_idx, score))
                
            if scored_sites:
                max_score = max(site[1] for site in scored_sites)
                self._matches = [site[0] for site in scored_sites if site[1] == max_score]

    def morph(self):
        if not self.original: return None
        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None: return None

        if not self._matches:
            return MolpherMol(other=rdkit_mol)

        c_idx = random.choice(self._matches)
        
        try:
            rw_mol = Chem.RWMol(rdkit_mol)
            
            new_o_idx = rw_mol.AddAtom(Chem.Atom(8))
            c_atom = rw_mol.GetAtomWithIdx(c_idx)
            o_atom = rw_mol.GetAtomWithIdx(new_o_idx)
            
            c_atom.SetNoImplicit(False)
            c_atom.SetNumExplicitHs(0)
            o_atom.SetNoImplicit(False)
            o_atom.SetNumExplicitHs(0)
            
            rw_mol.AddBond(c_idx, new_o_idx, Chem.BondType.SINGLE)
            
            new_mol = rw_mol.GetMol()
            
            new_mol.UpdatePropertyCache(strict=False)
            Chem.SanitizeMol(new_mol, Chem.SanitizeFlags.SANITIZE_ALL)
            Chem.AssignStereochemistry(new_mol, cleanIt=True, force=True)
            
            return MolpherMol(other=new_mol)
        except:
            return MolpherMol(other=rdkit_mol)

    def getName(self): return self._name

hydroxylation_op = AromaticHydroxylation()

gleevec_smiles = "CC1=C(C=CC(=C1)NC(=O)C2=CC=C(C=C2)CN3CCN(CC3)C)NC4=NC=CC(=N4)C5=CN=CC=C5"
mol_gleevec = MolpherMol(gleevec_smiles)
hydroxylation_op = AromaticHydroxylation()
hydroxylation_op.setOriginal(mol_gleevec)
product_gleevec = hydroxylation_op.morph()

print("=== GLEVEC HYDROXYLATION TEST ===")
print(f"SOURCE: {mol_gleevec.getSMILES()}")
print(f"TARGET: {product_gleevec.getSMILES() if product_gleevec else 'Failed'}")

=== GLEVEC HYDROXYLATION TEST ===
SOURCE: CC1=C(NC2=NC=CC(C3=CN=CC=C3)=N2)C=CC(NC(=O)C2=CC=C(CN3CCN(C)CC3)C=C2)=C1
TARGET: CC1=C(NC2=NC=C(O)C(C3=CN=CC=C3)=N2)C=CC(NC(=O)C2=CC=C(CN3CCN(C)CC3)C=C2)=C1
